# Car Price Baseline Pipeline

Pipeline này dùng `train.csv`, `val.csv`, `test.csv` đã split sẵn.

Mục tiêu:
1. Drop `url`, `date`, `price` vì target dùng `price_log`.
2. Thử nhiều cách encode categorical features: One-Hot, Ordinal/Label-like, Target Encoding.
3. Scale dữ liệu cho model cần scale.
4. Train nhiều baseline model và report kết quả trên train/val/test.

In [ ]:
# Nếu thiếu thư viện target encoding, chạy cell này trước
# !pip install category_encoders

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception:
    HAS_XGB = False

try:
    from lightgbm import LGBMRegressor
    HAS_LGBM = True
except Exception:
    HAS_LGBM = False

try:
    from catboost import CatBoostRegressor
    HAS_CATBOOST = True
except Exception:
    HAS_CATBOOST = False

try:
    import category_encoders as ce
    HAS_CE = True
except Exception:
    HAS_CE = False

In [ ]:
DATA_DIR = Path("../../data/bonbanh/split")

TRAIN_PATH = DATA_DIR / "train.csv"
VAL_PATH = DATA_DIR / "val.csv"
TEST_PATH = DATA_DIR / "test.csv"

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print(train_df.shape, val_df.shape, test_df.shape)
train_df.head()

In [ ]:
TARGET = "price_log"

DROP_COLS = [
    "url",
    "date",
    "price",      # bỏ vì target dùng price_log
]

DROP_COLS = [col for col in DROP_COLS if col in train_df.columns]

feature_cols = [col for col in train_df.columns if col not in DROP_COLS + [TARGET]]

X_train = train_df[feature_cols].copy()
y_train = train_df[TARGET].copy()

X_val = val_df[feature_cols].copy()
y_val = val_df[TARGET].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df[TARGET].copy()

print("Drop cols:", DROP_COLS)
print("Target:", TARGET)
print("Feature cols:", feature_cols)

In [ ]:
num_cols = X_train.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numerical:", num_cols)
print("Categorical:", cat_cols)
print("n_num:", len(num_cols), "n_cat:", len(cat_cols))

## 1. Preprocessing pipelines

Có 3 hướng encode:

- `onehot`: phù hợp với Linear Regression, Ridge, Lasso, KNN khi số category không quá lớn.
- `ordinal`: giống label encode cho nhiều cột categorical, thường dùng nhanh với tree models.
- `target`: encode category bằng trung bình target, hợp với biến cardinality cao như `name`, `model`, `trim`.

In [ ]:
def make_onehot_preprocessor(scale_numeric=True):
    num_steps = [
        ("imputer", SimpleImputer(strategy="median")),
    ]
    if scale_numeric:
        num_steps.append(("scaler", StandardScaler()))

    num_pipe = Pipeline(num_steps)

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])

    return ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ])


def make_ordinal_preprocessor(scale_numeric=False):
    num_steps = [
        ("imputer", SimpleImputer(strategy="median")),
    ]
    if scale_numeric:
        num_steps.append(("scaler", StandardScaler()))

    num_pipe = Pipeline(num_steps)

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ordinal", OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1
        ))
    ])

    return ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ])


def make_target_preprocessor(scale_numeric=False):
    if not HAS_CE:
        raise ImportError("Bạn cần cài category_encoders: pip install category_encoders")

    num_steps = [
        ("imputer", SimpleImputer(strategy="median")),
    ]
    if scale_numeric:
        num_steps.append(("scaler", StandardScaler()))

    num_pipe = Pipeline(num_steps)

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("target", ce.TargetEncoder())
    ])

    return ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ])

In [ ]:
def get_models():
    models = {
        "linear_regression": LinearRegression(),
        "ridge": Ridge(alpha=1.0, random_state=42),
        "lasso": Lasso(alpha=0.001, random_state=42, max_iter=5000),
        "elasticnet": ElasticNet(alpha=0.001, l1_ratio=0.5, random_state=42, max_iter=5000),

        "knn_5": KNeighborsRegressor(n_neighbors=5),
        "knn_15": KNeighborsRegressor(n_neighbors=15),

        "decision_tree": DecisionTreeRegressor(
            max_depth=None,
            min_samples_leaf=5,
            random_state=42
        ),
        "random_forest": RandomForestRegressor(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=2,
            n_jobs=-1,
            random_state=42
        ),
        "extra_trees": ExtraTreesRegressor(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=2,
            n_jobs=-1,
            random_state=42
        ),
        "gradient_boosting": GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=3,
            random_state=42
        ),
        "hist_gradient_boosting": HistGradientBoostingRegressor(
            max_iter=300,
            learning_rate=0.05,
            random_state=42
        ),
    }

    if HAS_XGB:
        models["xgboost"] = XGBRegressor(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            n_jobs=-1,
            random_state=42
        )

    if HAS_LGBM:
        models["lightgbm"] = LGBMRegressor(
            n_estimators=500,
            learning_rate=0.05,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1
        )

    return models

In [ ]:
def evaluate_regression(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }


def evaluate_pipeline(name, encoding, pipeline):
    pipeline.fit(X_train, y_train)

    rows = []

    for split_name, X, y in [
        ("train", X_train, y_train),
        ("val", X_val, y_val),
        ("test", X_test, y_test),
    ]:
        pred = pipeline.predict(X)
        metrics = evaluate_regression(y, pred)

        rows.append({
            "model": name,
            "encoding": encoding,
            "split": split_name,
            **metrics
        })

    return rows, pipeline

## 2. Chạy baseline models

Quy ước:
- Linear/KNN dùng `onehot + scale`.
- Tree/Boosting dùng `ordinal` hoặc `target`.
- Target encoding chỉ fit bằng `y_train`, sau đó transform val/test để tránh leakage.

In [ ]:
experiments = []

models = get_models()

linear_like = [
    "linear_regression",
    "ridge",
    "lasso",
    "elasticnet",
    "knn_5",
    "knn_15",
]

tree_like = [
    "decision_tree",
    "random_forest",
    "extra_trees",
    "gradient_boosting",
    "hist_gradient_boosting",
    "xgboost",
    "lightgbm",
]

# OneHot + Scale cho linear/KNN
for model_name in linear_like:
    if model_name not in models:
        continue

    pipe = Pipeline([
        ("preprocess", make_onehot_preprocessor(scale_numeric=True)),
        ("model", models[model_name])
    ])

    experiments.append((model_name, "onehot_scaled", pipe))


# Ordinal cho tree/boosting
for model_name in tree_like:
    if model_name not in models:
        continue

    pipe = Pipeline([
        ("preprocess", make_ordinal_preprocessor(scale_numeric=False)),
        ("model", models[model_name])
    ])

    experiments.append((model_name, "ordinal", pipe))


# Target Encoding cho tree/boosting
if HAS_CE:
    for model_name in tree_like:
        if model_name not in models:
            continue

        pipe = Pipeline([
            ("preprocess", make_target_preprocessor(scale_numeric=False)),
            ("model", models[model_name])
        ])

        experiments.append((model_name, "target", pipe))

print("Total experiments:", len(experiments))
[(m, e) for m, e, _ in experiments]

In [ ]:
all_rows = []
fitted_pipelines = {}

for model_name, encoding_name, pipe in experiments:
    exp_name = f"{model_name}__{encoding_name}"
    print("Running:", exp_name)

    try:
        rows, fitted_pipe = evaluate_pipeline(model_name, encoding_name, pipe)
        all_rows.extend(rows)
        fitted_pipelines[exp_name] = fitted_pipe

    except Exception as e:
        print("FAILED:", exp_name)
        print(type(e).__name__, e)

results = pd.DataFrame(all_rows)
results.head()

In [ ]:
val_report = (
    results[results["split"] == "val"]
    .sort_values("RMSE")
    .reset_index(drop=True)
)

val_report

In [ ]:
test_report = (
    results[results["split"] == "test"]
    .sort_values("RMSE")
    .reset_index(drop=True)
)

test_report

In [ ]:
full_report = (
    results
    .sort_values(["split", "RMSE"])
    .reset_index(drop=True)
)

OUT_DIR = DATA_DIR / "baseline_reports"
OUT_DIR.mkdir(parents=True, exist_ok=True)

results.to_csv(OUT_DIR / "all_results.csv", index=False, encoding="utf-8-sig")
val_report.to_csv(OUT_DIR / "val_report.csv", index=False, encoding="utf-8-sig")
test_report.to_csv(OUT_DIR / "test_report.csv", index=False, encoding="utf-8-sig")

print("Saved to:", OUT_DIR.resolve())

## 3. Chọn best model theo validation

Sau khi chọn model tốt nhất bằng validation, ta xem lại performance trên test.

In [ ]:
best_row = val_report.iloc[0]
best_name = f"{best_row['model']}__{best_row['encoding']}"

print("Best model by VAL RMSE:")
print(best_row)
print("Pipeline key:", best_name)

best_pipeline = fitted_pipelines[best_name]

In [ ]:
best_test_pred = best_pipeline.predict(X_test)

best_test_metrics = evaluate_regression(y_test, best_test_pred)
best_test_metrics

## 4. Error analysis đơn giản

In [ ]:
error_df = test_df.copy()
error_df["pred_price_log"] = best_test_pred
error_df["abs_error_log"] = np.abs(error_df[TARGET] - error_df["pred_price_log"])

cols = [
    "brand", "model", "name", "year", "price", "price_log",
    "pred_price_log", "abs_error_log"
]
cols = [c for c in cols if c in error_df.columns]

error_df[cols].sort_values("abs_error_log", ascending=False).head(30)

In [ ]:
brand_error = (
    error_df
    .groupby("brand")
    .agg(
        count=("brand", "size"),
        mae_log=("abs_error_log", "mean")
    )
    .sort_values("mae_log", ascending=False)
)

brand_error.head(30)

In [ ]:
model_error = (
    error_df
    .groupby(["brand", "model"])
    .agg(
        count=("model", "size"),
        mae_log=("abs_error_log", "mean")
    )
    .query("count >= 10")
    .sort_values("mae_log", ascending=False)
)

model_error.head(30)

## 5. Lưu best model

In [ ]:
import joblib

MODEL_DIR = DATA_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(best_pipeline, MODEL_DIR / "best_baseline_pipeline.joblib")

print("Saved:", (MODEL_DIR / "best_baseline_pipeline.joblib").resolve())